# 02 – Benchmark Iterations

> **Research prototype notebook.**  This notebook is a compact, reproducible
> benchmark across four synthetic demand scenarios comparing the THUNBIT
> detector variants.  Results are simulation-based and exploratory — they are
> not validated performance claims.

## What this notebook covers

1. Scenario generation (stable, mean-shift, variance-spike, gradual-drift)
2. Benchmark helper functions
3. Comparison across Baseline / V4.1 / V4.2 / V4.3
4. Summary tables — false-alert burden and break-detection speed
5. The qualitative iteration story

## Iteration story in brief

| Version | Key change | What it improved | What it didn't fix |
|---------|-----------|-----------------|-------------------|
| Old baseline | No state machine | Fast response | Very noisy on stable series |
| V4 | Hysteresis + smoothing + confirmation | Fewer alert clusters | Added detection delay |
| V4.1 | Cooldown + relaxed thresholds | Faster than V4 | Stable alert burden unchanged |
| V4.2 | Score normalization (median baseline) | **Major** false-alert reduction | Over-damped on real breaks |
| **V4.3** | Lower-quantile baseline + warmup | Recovers break speed | Stable alerts reduced but unsolved |

## Setup

```bash
pip install -e .   # from the repository root
```

In [ ]:
import numpy as np
import pandas as pd

from thunbit import (
    DemandStateDetector,
    StabilizedDemandDetectorV41,
    StabilizedDemandDetectorV42,
    StabilizedDemandDetectorV43,
)

# Reproducible seed for all scenarios in this notebook
BASE_SEED = 42
N         = 400   # days per series
BREAK_DAY = 200   # day of injected break
N_SEEDS   = 5     # seeds per scenario (keep small for speed; docs use 10)

print('Setup complete.')
print(f'Series length: {N} days,  break at day {BREAK_DAY},  {N_SEEDS} seeds per scenario.')

## 1. Scenario generators

We use four scenarios that test complementary aspects of detection:

| Scenario | Break type |
|----------|------------|
| `stable` | No break — measures false-alert burden |
| `mean_shift` | Demand mean increases ~60% at day 200 |
| `variance_spike` | Demand variance multiplied by ~7× at day 200 |
| `gradual_drift` | Demand mean decays by ~25% over 60 days starting at day 200 |

A full benchmark (see `docs/benchmarking.md`) also includes cycle-break and
intermittent scenarios.  They are omitted here for brevity.

In [ ]:
def make_stable(rng, n=400):
    """Stationary normal demand – no injected break."""
    return rng.normal(loc=100.0, scale=10.0, size=n).clip(0)


def make_mean_shift(rng, n=400, break_day=200):
    """Demand mean jumps from 100 to 160 at break_day."""
    pre  = rng.normal(loc=100.0, scale=10.0, size=break_day).clip(0)
    post = rng.normal(loc=160.0, scale=12.0, size=n - break_day).clip(0)
    return np.concatenate([pre, post])


def make_variance_spike(rng, n=400, break_day=200):
    """Demand variance increases ~7× at break_day; mean unchanged."""
    pre  = rng.normal(loc=100.0, scale=10.0, size=break_day).clip(0)
    post = rng.normal(loc=100.0, scale=70.0, size=n - break_day).clip(0)
    return np.concatenate([pre, post])


def make_gradual_drift(rng, n=400, break_day=200, drift_window=60):
    """Demand mean gradually decays from 100 to 75 over drift_window days."""
    pre   = rng.normal(loc=100.0, scale=10.0, size=break_day).clip(0)
    means = np.concatenate([
        np.linspace(100.0, 75.0, drift_window),
        np.full(n - break_day - drift_window, 75.0),
    ])
    post  = rng.normal(loc=means, scale=10.0).clip(0)
    return np.concatenate([pre, post])


SCENARIO_FUNCS = {
    'stable':         make_stable,
    'mean_shift':     make_mean_shift,
    'variance_spike': make_variance_spike,
    'gradual_drift':  make_gradual_drift,
}

print('Scenario generators defined.')

## 2. Benchmark helpers

Two metric functions:
- `stable_metrics` – measures false-alert burden on a no-break series
- `break_metrics`  – measures detection speed on a break-injected series

In [ ]:
def run_detector(det, series):
    """Run the appropriate detect method for a detector instance."""
    if hasattr(det, 'detect_rolling_stabilized'):
        return det.detect_rolling_stabilized(series)
    return det.detect_rolling(series)


def count_clusters(states):
    """Count distinct alert clusters (runs of non-STABLE states)."""
    count = 0
    in_cluster = False
    for s in states:
        if s != 'STABLE':
            if not in_cluster:
                count += 1
                in_cluster = True
        else:
            in_cluster = False
    return count


def stable_metrics(df):
    """Metrics for a no-break (stable) scenario run."""
    n = len(df)
    alert_days = (df['state'] != 'STABLE').sum()
    return {
        'any_alert': int(alert_days > 0),
        'alert_days_pct': alert_days / n,
        'fp_clusters': count_clusters(df['state'].tolist()),
    }


def break_metrics(df, break_day, min_run=3):
    """Metrics for a break-injected scenario run."""
    pre_df  = df[df['t'] < break_day]
    post_df = df[df['t'] >= break_day]

    fp_clusters = count_clusters(pre_df['state'].tolist())

    # First sustained alert: min_run consecutive non-STABLE rows
    alert_day = None
    post_states = post_df['state'].tolist()
    post_t      = post_df['t'].tolist()
    for i in range(len(post_states) - min_run + 1):
        window = post_states[i: i + min_run]
        if all(s != 'STABLE' for s in window):
            alert_day = post_t[i]
            break

    detected = int(alert_day is not None)
    days_late = (alert_day - break_day) if detected else None
    return {
        'detected': detected,
        'days_late': days_late,
        'fp_clusters': fp_clusters,
    }


print('Benchmark helpers defined.')

## 3. Detector registry

In [ ]:
DETECTORS = {
    'Baseline': DemandStateDetector(),
    'V4.1':     StabilizedDemandDetectorV41(),
    'V4.2':     StabilizedDemandDetectorV42(),
    'V4.3':     StabilizedDemandDetectorV43(),
}

print('Detectors ready:', list(DETECTORS.keys()))

## 4. Run benchmark

We run each detector on each scenario for `N_SEEDS` random seeds and collect
the metrics defined above.  This takes a few seconds.

In [ ]:
all_results = []   # list of dicts, one per (scenario, detector, seed)

for scenario_name, gen_fn in SCENARIO_FUNCS.items():
    for seed in range(BASE_SEED, BASE_SEED + N_SEEDS):
        rng = np.random.default_rng(seed)
        series = gen_fn(rng, n=N, **({'break_day': BREAK_DAY} if scenario_name != 'stable' else {}))

        for det_name, det in DETECTORS.items():
            df = run_detector(det, series)

            if scenario_name == 'stable':
                m = stable_metrics(df)
                m['scenario'] = scenario_name
                m['detector'] = det_name
                m['seed']     = seed
            else:
                m = break_metrics(df, BREAK_DAY)
                m['scenario'] = scenario_name
                m['detector'] = det_name
                m['seed']     = seed

            all_results.append(m)

results_df = pd.DataFrame(all_results)
print(f'Collected {len(results_df)} result rows.')
print(results_df.head())

## 5. False-alert burden on stable series

The stable scenario has no injected break; any non-STABLE days are false alerts.
Lower values are better.

In [ ]:
stable_results = results_df[results_df['scenario'] == 'stable'].copy()

stable_summary = (
    stable_results
    .groupby('detector')
    .agg(
        any_alert_rate  = ('any_alert',       'mean'),
        mean_alert_pct  = ('alert_days_pct',  'mean'),
        mean_fp_clusters= ('fp_clusters',     'mean'),
    )
    .reindex(['Baseline', 'V4.1', 'V4.2', 'V4.3'])
    .reset_index()
)

stable_summary['mean_alert_pct'] = stable_summary['mean_alert_pct'].map('{:.1%}'.format)
stable_summary['any_alert_rate'] = stable_summary['any_alert_rate'].map('{:.2f}'.format)
stable_summary['mean_fp_clusters'] = stable_summary['mean_fp_clusters'].map('{:.1f}'.format)

print('Stable-series false-alert summary')
print(f'(N_SEEDS={N_SEEDS}, series length {N} days, synthetic data)')
print()
print(stable_summary.to_string(index=False))
print()
print('Lower any_alert_rate, mean_alert_pct, and mean_fp_clusters = fewer false alerts.')
print('V4.2 and V4.3 show the largest reductions.')

## 6. Break-detection speed

For each break scenario we measure:
- `detection_rate` — fraction of seeds where the break was eventually detected
- `mean_days_late` — mean delay between break day and first sustained alert
- `mean_fp_clusters` — mean false-alert clusters *before* the break

Lower `mean_days_late` = faster detection.  But this must be read alongside the
false-alert burden above.

In [ ]:
break_scenarios = [s for s in SCENARIO_FUNCS if s != 'stable']
break_results   = results_df[results_df['scenario'].isin(break_scenarios)].copy()

break_summary = (
    break_results
    .groupby(['scenario', 'detector'])
    .agg(
        detection_rate   = ('detected',    'mean'),
        mean_days_late   = ('days_late',   'mean'),
        mean_fp_clusters = ('fp_clusters', 'mean'),
    )
    .reset_index()
)

# Format
break_summary['detection_rate']   = break_summary['detection_rate'].map('{:.2f}'.format)
break_summary['mean_days_late']   = break_summary['mean_days_late'].apply(
    lambda x: f'{x:.1f}' if pd.notna(x) else 'N/A'
)
break_summary['mean_fp_clusters'] = break_summary['mean_fp_clusters'].map('{:.1f}'.format)

for sc in break_scenarios:
    sub = break_summary[break_summary['scenario'] == sc].set_index('detector').reindex(
        ['Baseline', 'V4.1', 'V4.2', 'V4.3']
    ).reset_index()
    print(f'--- {sc} (break at day {BREAK_DAY}) ---')
    print(sub[['detector', 'detection_rate', 'mean_days_late', 'mean_fp_clusters']].to_string(index=False))
    print()

## 7. Combined tradeoff view

The core design tension:

> More suppression → fewer stable-series false clusters → more delay on real breaks.

The table below shows the tradeoff for the mean-shift scenario, the clearest
test case.

In [ ]:
# Pull mean_shift detection speed
ms_speed = break_summary[break_summary['scenario'] == 'mean_shift'][['detector', 'mean_days_late']].copy()
ms_speed = ms_speed.rename(columns={'mean_days_late': 'mean_shift_days_late'})

# Pull stable false-alert burden
stable_burden = stable_summary[['detector', 'mean_alert_pct', 'mean_fp_clusters']].copy()

combined = stable_burden.merge(ms_speed, on='detector')
combined = combined.reindex(
    combined.index[combined['detector'].map({'Baseline': 0, 'V4.1': 1, 'V4.2': 2, 'V4.3': 3}).argsort()]
).reset_index(drop=True)

print('False-alert burden vs break-detection speed (mean_shift scenario):')
print(combined.to_string(index=False))
print()
print('Interpretation:')
print('  Baseline / V4.1 : fast detection, high false-alert burden')
print('  V4.2            : very low false-alert burden, slow detection (over-damped)')
print('  V4.3            : best current compromise — reduced burden, acceptable speed')

## 8. Iteration story commentary

The progression from Baseline → V4.3 maps to a concrete design journey:

### Old baseline
Direct threshold comparison on raw confidence.  Fast, but every seed of the
stable scenario triggers alerts.  `mean_alert_days_pct` ≈ 32–47% depending on
the seed set.

### V4 / V4.1 — state-machine stabilization
Adding hysteresis, smoothing, confirmation, and cooldown (V4 and V4.1) reduced
the number of distinct false-alert *clusters* and recovered detection speed that
V4 had sacrificed.  But the underlying raw-confidence score still has a
systematic positive bias on stationary series, so the *duration* of false-alert
periods remained largely unchanged.  The state machine helped with noisy
toggling; it did not fix the calibration problem.

### V4.2 — score normalization breakthrough
V4.2 shifted the approach fundamentally: instead of filtering the output of a
mis-calibrated score, it normalized the score itself against the SKU's own
recent baseline.  The result was a major reduction in stable-series false
alerts.  The cost was over-damping: the normalized score adapts to genuine
breaks quickly enough that the excess signal is suppressed before the state
machine can fire.

### V4.3 — current best compromise
V4.3 addresses V4.2's over-damping by:
1. Using the 25th percentile instead of the median as the baseline.  The lower
   quantile stays closer to the true noise floor and adapts more slowly when a
   genuine break raises confidence, so break signals produce a larger, longer
   excess.
2. Adding warmup suppression to prevent false alerts during the initial 28
   output rows when the baseline history is thin.

V4.3 is the current best experimental operating point.  It is not a final
solution.  Score calibration — not just state-transition logic — remains the
central open design challenge.

In [ ]:
# Optional: visualize the tradeoff as a scatter plot (requires matplotlib)
try:
    import matplotlib.pyplot as plt

    # Numeric forms for plotting
    stable_burden_num = (
        results_df[results_df['scenario'] == 'stable']
        .groupby('detector')['alert_days_pct']
        .mean()
        .reindex(['Baseline', 'V4.1', 'V4.2', 'V4.3'])
    )
    ms_delay_num = (
        results_df[results_df['scenario'] == 'mean_shift']
        .groupby('detector')['days_late']
        .mean()
        .reindex(['Baseline', 'V4.1', 'V4.2', 'V4.3'])
    )

    fig, ax = plt.subplots(figsize=(7, 5))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    for (det, fa, dl), c in zip(
        zip(stable_burden_num.index, stable_burden_num.values, ms_delay_num.values),
        colors
    ):
        ax.scatter(fa * 100, dl if pd.notna(dl) else 0,
                   s=120, color=c, zorder=5, label=det)
        ax.annotate(det, (fa * 100, dl if pd.notna(dl) else 0),
                    textcoords='offset points', xytext=(6, 4), fontsize=9)

    ax.set_xlabel('Stable-series alert days (%) — lower is better', fontsize=10)
    ax.set_ylabel('Mean days late on mean-shift — lower is better', fontsize=10)
    ax.set_title('Detection tradeoff: false-alert burden vs break-detection speed\n'
                 '(synthetic data, illustrative)', fontsize=11)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig('tradeoff_scatter.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Plot saved to tradeoff_scatter.png')

except ImportError:
    print('matplotlib not installed – skipping tradeoff plot.')
    print('Run:  pip install matplotlib')

## Caveats

- These results are from `N_SEEDS` seeds.  The `docs/benchmarking.md` reference
  tables used 10 seeds; numbers will differ slightly here.
- All demand series are synthetic.  Behaviour on real SKU data is unknown.
- Detection delay numbers depend on the `min_run=3` sustained-alert criterion;
  different criteria yield different numbers.
- No automated parameter tuning was done; all thresholds were set manually.

See `docs/benchmarking.md` for the reference tables and `docs/limitations.md`
for the full list of open problems.

**Next:** `03_cost_simulation.ipynb` — illustrative business-cost framing.